## Extract

In [1]:
# import des fonctions d'extraction et transformation
import importlib
import sys
if 'etl.config' in sys.modules:
    del sys.modules['etl.config']
if 'etl.transform' in sys.modules:
    del sys.modules['etl.transform']
if 'etl.extract' in sys.modules:
    del sys.modules['etl.extract']
    
from etl.extract import extract_charges_telephoniques, extract_charges_impression
from etl.transform import transform_charges_telephoniques, transform_charges_impression
from etl.config import persist_clean_tables

In [2]:
# Extraction + Transformation des données en une seule étape
ChargesTelephoniques = extract_charges_telephoniques()
ChargesImpression = extract_charges_impression()
ChargesTelephoniques = transform_charges_telephoniques(ChargesTelephoniques)
ChargesImpression = transform_charges_impression(ChargesImpression)

print(f"ChargesTelephoniques : {ChargesTelephoniques.shape}")
print(f"ChargesImpression   : {ChargesImpression.shape}")

[INFO] Validation dates ChargesTelephoniques - colonne 'DateOperation'...
  ✓ Toutes les dates valides (4945 lignes)
[INFO] Propagation NomRole et NumeroTelephone (dernier jour du mois → tous les jours du mois)...
  ✓ 4945 combinaison(s) traité(e)s
[INFO] Validation dates ChargesTelephoniques - colonne 'DateOperation'...
  ✓ Toutes les dates valides (4945 lignes)
[INFO] Imputation NomRole par ForfaitTND...
  Lignes avec NomRole == 'INCONNU' : 1319
  Forfaits mappés : 1142
  Distribution des forfaits non mappés :
    - ForfaitTND 60: 147 lignes
    - ForfaitTND 15: 19 lignes
    - ForfaitTND 45: 11 lignes
  ✓ 1142 NomRole imputés par ForfaitTND
[INFO] Normalisation NomDepartement (ChargesTelephoniques) - propagation dernière valeur valide du mois...
  ✓ 3265 combinaison(s) traité(e)s
  ⚠️  1680 ligne(s) avec département invalide → 'INCONNU'
[INFO] Normalisation CodeEmployee (format: YAZ+nombre ou INCONNU)...
  ✓ CodeEmployee reformatés
[INFO] Uniformisation NomDepartement (le plus récur

In [3]:
# Affichage des premières lignes du DataFrame ChargesTelephoniques
ChargesTelephoniques.head()

,TelephoniqueID,NomDepartement,CodeDepartement,DateOperation,NomRole,CodeRole,CodeEmployee,NumeroTelephone,ForfaitTND,DateValid
0,1,ACHAT,ACH,2023-01-31,TECHNICIEN,TECH,YAZ054,90941122,25,True
1,2,LOGISTIQUE,LOG,2023-01-31,SPECIALISTE,SP,YAZ080,50809900,40,True
2,3,LOGISTIQUE,LOG,2023-01-31,SUPERVISEUR,SUP,YAZ079,20798899,50,True
3,4,ENGENIERIE,ENG,2023-01-31,HEAD,HD,YAZ077,50576677,0,True
4,5,ENGENIERIE,ENG,2023-01-31,TECHNICIEN,TECH,YAZ075,90354455,25,True


In [4]:
# affichages des premières lignes du DataFrame ChargesImpression
ChargesImpression.head()

,ImpressionID,NomDepartement,CodeDepartement,DateImpression,TypeImpression,NbPages,CoutUnitaire,FormatPapier,CouleurImpression,DateValid
0,1,EHS,EHS,2023-01-01,A4-COULEUR,10,0.156,A4,COULEUR,True
1,2,PRODUCTION B,PROD-B,2023-01-01,A4-NB,83,0.026,A4,NOIR ET BLANC,True
2,3,PRODUCTION B,PROD-B,2023-01-01,A4-COULEUR,157,0.156,A4,COULEUR,True
3,4,PRODUCTION A,PROD-A,2023-01-01,A3-NB,151,0.052,A3,NOIR ET BLANC,True
4,5,QUALITE,QUA,2023-01-01,A4-COULEUR,117,0.156,A4,COULEUR,True


## Transform

In [5]:
# Affichage des données transformées - ChargesTelephoniques

In [6]:
print(f"ChargesTelephoniques apres transformation : {ChargesTelephoniques.shape}")
ChargesTelephoniques.head()

ChargesTelephoniques apres transformation : (4945, 10)


,TelephoniqueID,NomDepartement,CodeDepartement,DateOperation,NomRole,CodeRole,CodeEmployee,NumeroTelephone,ForfaitTND,DateValid
0,1,ACHAT,ACH,2023-01-31,TECHNICIEN,TECH,YAZ054,90941122,25,True
1,2,LOGISTIQUE,LOG,2023-01-31,SPECIALISTE,SP,YAZ080,50809900,40,True
2,3,LOGISTIQUE,LOG,2023-01-31,SUPERVISEUR,SUP,YAZ079,20798899,50,True
3,4,ENGENIERIE,ENG,2023-01-31,HEAD,HD,YAZ077,50576677,0,True
4,5,ENGENIERIE,ENG,2023-01-31,TECHNICIEN,TECH,YAZ075,90354455,25,True


In [7]:
print(f"ChargesImpression apres transformation : {ChargesImpression.shape}")
ChargesImpression.head()

ChargesImpression apres transformation : (9945, 10)


,ImpressionID,NomDepartement,CodeDepartement,DateImpression,TypeImpression,NbPages,CoutUnitaire,FormatPapier,CouleurImpression,DateValid
0,1,EHS,EHS,2023-01-01,A4-COULEUR,10,0.156,A4,COULEUR,True
1,2,PRODUCTION B,PROD-B,2023-01-01,A4-NB,83,0.026,A4,NOIR ET BLANC,True
2,3,PRODUCTION B,PROD-B,2023-01-01,A4-COULEUR,157,0.156,A4,COULEUR,True
3,4,PRODUCTION A,PROD-A,2023-01-01,A3-NB,151,0.052,A3,NOIR ET BLANC,True
4,5,QUALITE,QUA,2023-01-01,A4-COULEUR,117,0.156,A4,COULEUR,True


## Stockage SQL Server

In [8]:
from etl.config import persist_clean_tables

# Créer la base SQL Server réutilisable avec les deux tables nettoyées
persist_clean_tables(ChargesTelephoniques, ChargesImpression)

[LOAD] Création de la base nettoyée et persistance des tables...
  ✓ Base 'Yazaki_Clean' prête avec 2 table(s) nettoyée(s)
  ✓ ChargesTelephoniques: (4945, 10)
  ✓ ChargesImpression   : (9945, 10)


## Load

In [9]:
# import de la fonction de chargement
from etl.load import load_all

In [10]:
# Chargement des données dans le Data Warehouse
load_all(ChargesTelephoniques, ChargesImpression)


[PIPELINE] LOAD - Chargement des données dans DW_Yazaki

[LOAD] Suppression des données existantes...
  ✓ Tables vidées (Dim_Temps incluse)
[LOAD] Chargement Dim_Departement...
  ✓ 16 département(s) chargé(s)
[LOAD] Chargement Dim_Role...
  ✓ 11 rôle(s) chargé(s)
[LOAD] Chargement Dim_Employee...
  ✓ 131 employé(s) chargé(s)
[LOAD] (Re)chargement Dim_Temps (DateID = 1..n)...
  ✓ Dim_Temps chargée: 1247 date(s) (2023-01-01 → 2026-05-31)
[LOAD] Chargement Dim_Impression...
  ✓ 4 type(s) d'impression chargé(s)
[LOAD] Chargement Fact_Telephone...
  ✓ 4945 ligne(s) chargée(s) dans Fact_Telephone
[LOAD] Chargement Fact_Impression...
  ✓ 9945 ligne(s) chargée(s) dans Fact_Impression

[SUCCÈS] Chargement terminé avec succès dans DW_Yazaki !



In [11]:
ChargesImpression[ChargesImpression['NomDepartement'] == 'INCONNU']

,ImpressionID,NomDepartement,CodeDepartement,DateImpression,TypeImpression,NbPages,CoutUnitaire,FormatPapier,CouleurImpression,DateValid


In [12]:
ChargesImpression['NomDepartement'].unique()

<StringArray>
[         'EHS', 'PRODUCTION B', 'PRODUCTION A',      'QUALITE',
        'ACHAT',           'RH',   'ENGENIERIE',   'LOGISTIQUE',
        'COSEE',           'IT',          'OLS',           'TD',
    'DIRECTION',          'NYS',         'PLPP',      'FINANCE']
Length: 16, dtype: string